# HDFS Log Exploration & Pipeline Walkthrough

Exploratory analysis of the HDFS_2k dataset, plus an end-to-end walkthrough
of every stage this project's pipeline runs: preprocessing -> features ->
anomaly detection -> RAG retrieval -> full agent investigation.

**Run this from the project root** (`jupyter notebook notebooks/hdfs_analysis.ipynb`),
after putting `HDFS_2k.log_structured.csv` (and optionally the raw log and
templates file) in `data/raw/hdfs/`.

This notebook is exploratory / for-your-own-investigation use - the actual
production pipeline is `scripts/prepare_data.py`, `scripts/train_model.py`,
`scripts/build_index.py`, and the FastAPI service in `src/api/`.

In [ ]:
import sys
from pathlib import Path

# Run from notebooks/, so the project root is one level up.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 100)
%matplotlib inline

## 1. Load the raw structured logs

Uses the same `DataLoader` the rest of the pipeline uses - no separate
loading logic here.

In [ ]:
from src.data import DataLoader

loader = DataLoader(ROOT / "data" / "raw" / "hdfs")
raw = loader.load_structured_logs()
print(f"{len(raw):,} log lines")
raw.head()

## 2. Basic exploration

In [ ]:
raw["Level"].value_counts()

In [ ]:
raw["Component"].value_counts().head(10)

In [ ]:
raw["Component"].value_counts().head(10).plot(kind="barh", title="Top 10 components by log volume")
plt.xlabel("log lines")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Preprocess into time-windowed features

`DataPreprocessor` cleans the logs, parses HDFS's `YYMMDD`/`HHMMSS` Date+Time
columns, normalizes log levels, extracts 5 incident signals (heartbeat,
block, replication, network, disk), and aggregates into windows. Adjust
`window` if your data doesn't span enough time for `1min` windows to be
meaningful (see `docs/evaluation.md` for why this matters).

In [ ]:
from src.data import DataPreprocessor

preprocessor = DataPreprocessor(window="1min")
logs = preprocessor.preprocess_logs(raw)
features = preprocessor.create_features(logs)
print(f"{len(logs):,} usable log rows -> {len(features):,} time windows")
features.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
features.plot(x="time_window", y="logs_per_window", ax=axes[0], legend=False, title="Log volume per window")
features.plot(x="time_window", y="error_count", ax=axes[1], legend=False, title="Errors per window", color="crimson")
plt.tight_layout()
plt.show()

## 4. Anomaly detection

Loads the trained model if `models/anomaly_model.pkl` already exists
(via `scripts/train_model.py`); otherwise trains one here on the features
above, purely for this exploration - the real training artifact should
still come from the script, so it's reproducible and versioned.

In [ ]:
from src.ml import AnomalyDetector

model_path = ROOT / "models" / "anomaly_model.pkl"
if model_path.exists():
    detector = AnomalyDetector().load(model_path)
    print(f"Loaded trained model from {model_path}")
else:
    print("No trained model found - training one here for exploration "
          "(run scripts/train_model.py to produce the real artifact).")
    detector = AnomalyDetector(contamination=0.05).train(features)

scored = pd.concat([features, detector.predict_with_scores(features)], axis=1)
scored.sort_values("anomaly_score", ascending=False).head(10)[
    ["time_window", "logs_per_window", "error_count", "heartbeat_failures", "anomaly_score", "is_anomaly"]
]

In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(scored["time_window"], scored["anomaly_score"])
plt.axhline(0.5, color="crimson", linestyle="--", label="anomaly threshold")
plt.title("Anomaly score over time")
plt.ylabel("anomaly_score")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Why was a window flagged? (SHAP explainability)

Picks the single highest-scored window from above and breaks down which
features actually drove that score.

In [ ]:
from src.explainability import IncidentExplainer

explainer = IncidentExplainer(detector)
top_window = scored.sort_values("anomaly_score", ascending=False).iloc[0]
explanation = explainer.explain(top_window)

print(f"Window: {top_window['time_window']}  |  score: {explanation['anomaly_score']:.3f}  "
      f"|  method: {explanation['method']}")
pd.DataFrame(explanation["top_factors"]).head(8)

## 6. RAG retrieval over the troubleshooting runbooks

Requires a built index (`scripts/build_index.py`) - this needs internet
access once, to download the embedding model.

In [ ]:
from src.rag import load_retriever

def _fmt(x):
    """Not every result has a dense_distance - a chunk can be retrieved via
    BM25 alone (outside the dense candidate set), in which case it's None."""
    return f"{x:.3f}" if x is not None else "n/a"

index_path = ROOT / "vector_db"
if (index_path / "index.faiss").exists():
    retriever = load_retriever(index_path)
    for r in retriever.retrieve_with_scores("DataNode heartbeat failure", top_k=3):
        print(f"[{r['rank']}] {r['source']}  (bm25={r['bm25_score']:.2f}, dense_dist={_fmt(r['dense_distance'])})")
        print("   ", r["content"][:150].replace("\n", " "), "...")
else:
    print(f"No index found at {index_path} - run: python scripts/build_index.py")

## 7. Full agent investigation

Runs the same investigation `POST /investigate` would, directly against the
in-memory objects built above - useful for iterating without restarting the
API. `IncidentService` numbers each time window as one incident
(`INC-001`, `INC-002`, ...), chronologically.

In [ ]:
from types import SimpleNamespace
from src.services import IncidentService
from src.agents import AgentTools, AgentPlanner, IncidentAgent

class _NotebookLoader:
    """Reuses the raw DataFrame already loaded above, instead of re-reading the CSV."""
    def load_structured_logs(self):
        return raw.copy()

incident_service = IncidentService(_NotebookLoader(), preprocessor=preprocessor)
predictor = SimpleNamespace(
    detector=detector,
    predict=lambda X: [
        {"anomaly_score": float(s), "is_anomaly": bool(a)}
        for s, a in zip(detector.score(X), detector.predict(X))
    ],
)
retriever_for_agent = retriever if (index_path / "index.faiss").exists() else SimpleNamespace(
    retrieve=lambda query, top_k=3: []  # degrades gracefully, same as the real API when no index is built
)

tools = AgentTools(incident_service, predictor, retriever_for_agent)
agent = IncidentAgent(AgentPlanner(), tools, explainer=explainer)

most_severe = incident_service.list_incidents(limit=1, sort_by="severity")[0]["incident_id"]
report = agent.investigate(most_severe)

print(f"Incident: {report['incident_id']}")
print(f"Root cause: {report['root_cause']}  (confidence {report['confidence']:.0%})")
print("\nEvidence:")
for e in report["evidence"]:
    print(" -", e)
print("\nRecommended actions:")
for i, r in enumerate(report["recommendations"], 1):
    print(f" {i}. {r}")

## Next steps

- Compare these exploratory numbers against `evaluation/model_evaluation.py`
  and `evaluation/rag_evaluation.py`'s more rigorous, repeatable reports
  (see `docs/evaluation.md`).
- To serve this as an actual service instead of a notebook: `scripts/prepare_data.py`
  -> `scripts/train_model.py` -> `scripts/build_index.py` -> `uvicorn src.api.main:app`.